# MNISym_coreg_regression

**Pipeline**

Using full-image coregistered anatomicals with their segementations normalized to MNI Symmetric template:

- run regression on each of: wm, gm, T1, csf

- for subjects with left hemisphere lesion, flip (along x-axis, L-R flip) their slope image
    
    - slope image for each of wm, gm, T1, csf

- get the average image for patients and controls

In [ ]:
# need path to root directory
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

In [ ]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl

import nitools as nt
from smarts_cerebellum import regression
from image_processing import overall_image
from image_processing import mirror_lesion

from pathlib import Path
import os

In [ ]:
# directories
base_dir = '/cifs/diedrichsen/data/smarts_cerebellum'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

## Run regression on each of: gm, wm, T1 anats

In [ ]:
def subj_unique_regression_MNISym_coreg(type):

    """
    Function for local use.

    Function to perform regression on coregistered and normalized (to MNISym template) images

    Input:
        type (str): type of image; used for finding image and for saving images
            valid types: GM, WM, T1
            TBA: CSF
    """

    for subj in p_df['subj_id'].unique():

        # find each subject's reference image, and run it through the regression
        refT1 = (p_df.loc[(p_df['subj_id']==subj), 'RefT1'].iloc[0]).strip()
        ref_img = f'{base_dir}/MNISym_{type}/{subj}/{subj}_{refT1}_MNISym_{type}_coreg_reslice.nii.gz'

        print(f"Regression on {subj} \n")

        intercept_img, slope_img = regression.perform_regression_week(subj_id = subj,
                            reference_img = ref_img
                            )
        

        # if images exist, save them
        if intercept_img is not None and slope_img is not None:
            results_path = f'{base_dir}/Regression/{subj}'
            results_path = Path(results_path)

            # comment this out if this directory already exists
            #results_path.mkdir(parents = True, exist_ok = True)

            nib.save(intercept_img, f'{results_path}/{subj}_MNISym_{type}_coreg_reslice_intercept.nii.gz')
            nib.save(slope_img, f'{results_path}/{subj}_MNISym_{type}_coreg_reslice_slope.nii.gz')


# name will be like <subj_id>_MNISym_GM_reslice_<alg = intercept/slope>.nii.gz

In [ ]:
# regression on GM segmentations
subj_unique_regression_MNISym_coreg(type = 'GM')

In [ ]:
# regression on WM segmentations
subj_unique_regression_MNISym_coreg(type = 'WM')

In [ ]:
# regression on normalized T1 anats
subj_unique_regression_MNISym_coreg(type = 'T1')

## GM slope: mirror + average_image

### Mirror

In [ ]:
# which side should I flip to? Find out which side most patients have a lesion, and choose that as our side to have lesion on

patients_df = p_df[p_df.isPatient==1]

# number of patients with lesion on..

# left hemisphere
num_left = len(patients_df[patients_df.LesionSide=='left '].subj_id.unique()) # note that whitespace after "left"

# right hemisphere
num_right = len(patients_df[patients_df.LesionSide=='right'].subj_id.unique())

print(f'Num left lesion: {num_left} \nNum right lesion: {num_right}')

In [ ]:
def subj_unique_flip(df, image_suffix, output_suffix = 'FlipLR'):
    """
    Flips specified image along the x-axis

    Image will be saved as {subj}_{image_suffix}_{output_suffix}.nii.gz
    """

    for subj in df['subj_id'].unique():
        image_to_flip = f'{base_dir}/Regression/{subj}/{subj}_{image_suffix}.nii.gz'

        if not Path(image_to_flip).exists():
            print(f'path does not exist for {subj}; skipping')
            continue
        
        flipped_img = mirror_lesion.FlipLR(image_to_flip)
        print(f'Flipped image for {subj}')

        # save flipped image
        nib.save(flipped_img, f'{base_dir}/Regression/{subj}/{subj}_{image_suffix}_{output_suffix}.nii.gz')
        

In [ ]:
# flip: slope image for coreg MNISym GM
left_lesion_df = patients_df[patients_df.LesionSide=='left ']
subj_unique_flip(left_lesion_df, image_suffix = 'MNISym_GM_coreg_reslice_slope')

### Average image

In [ ]:
patients_df = p_df[p_df.isPatient==1]
control_df = p_df[p_df.isPatient==0]

In [ ]:
# use affine from MNISym template

template_path = '/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/tpl-MNI152NLin2009cSymC_T1w.nii'
template_img = nib.load(template_path)
template_affine = template_img.affine

In [ ]:
patients_MNISym_GM_slope_right_average = overall_image.average_image(df = patients_df,
                                                         affine = template_affine,
                                                         suffix = 'MNISym_GM_coreg_reslice_slope')

In [ ]:
nib.save(patients_MNISym_GM_slope_right_average, f'{base_dir}/Regression/patients_MNISym_GM_slope_right_average.nii.gz')